# Supplementary Figure 12 — mining-suitability model performance by region

Fifteen regional panels in a compact **3-row × 5-column** layout at **183 × 114.8 mm**.
Each panel shows **AUC / F1 / Cohen's κ**: bars are fold means, error bars are
**±1 sample SD (ddof=1)**, and transparent hollow circles show the individual fold scores.
The upper-right label gives the region and actual valid-fold count, e.g. **ME (n=5)**.
EA has three valid folds; the other regions have five. Counts are read from the data.

Plotting data: `results/figure-data/Supplementary-Figure-12.csv`, produced by
`2-mining-suitability-model/4-train-models.ipynb` using per-region GroupKFold or
Leave-One-Group-Out evaluation. This notebook reads the saved scores without training.

Outputs: `figures/Supplementary-Figure-12.jpg`, `.png` (600 dpi), and `.svg`
(editable text and vector artwork). Export retains the exact physical canvas size.

Run from the repository root or this notebook's directory, using the activated
`wty` kernel (required for the project's NumPy/LAPACK environment).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.colors import to_rgba


In [ ]:
# Support execution from the notebook directory or the repository root.
ROOT = Path.cwd().resolve()
if not (ROOT / 'results' / 'figure-data' / 'Supplementary-Figure-12.csv').is_file():
    ROOT = ROOT.parent
FIGURE_DATA = ROOT / 'results' / 'figure-data' / 'Supplementary-Figure-12.csv'
FIGURES = ROOT / 'figures'
if not FIGURE_DATA.is_file():
    raise FileNotFoundError(f'Cannot find the saved fold scores: {FIGURE_DATA}')

# Nature double-column width; preserve this exact canvas size on export.
# https://research-figure-guide.nature.com/figures/building-and-exporting-figure-panels/
WIDTH_MM = 183
LEFT_MM, RIGHT_MM, TOP_MM, BOTTOM_MM = 11, 2, 2, 6
COLUMN_GAP_MM, ROW_GAP_MM = 3, 6
PANEL_MM = (WIDTH_MM - LEFT_MM - RIGHT_MM - 4 * COLUMN_GAP_MM) / 5
HEIGHT_MM = TOP_MM + BOTTOM_MM + 3 * PANEL_MM + 2 * ROW_GAP_MM

REGIONS = {
    1: "ME", 2: "LAC", 3: "NAF", 4: "WAF", 5: "EAF",
    6: "CAF", 7: "SAF", 8: "EU", 9: "SEA", 10: "OC",
    11: "SA", 12: "EA", 13: "CNA", 14: "NA", 15: "CHN",
}
METRICS = ["auc", "f1", "kappa"]
LABELS = ["AUC", "F1", "Cohen's κ"]
COLORS = ["#3B7FB6", "#E8843C", "#5BA85B"]

data = pd.read_csv(FIGURE_DATA).sort_values(["region", "fold"])
if data.duplicated(["region", "fold"]).any():
    raise ValueError("Expected one result per region and fold.")
if set(data["region"]) != set(REGIONS):
    raise ValueError("Expected results for all 15 regions.")
if not np.isfinite(data[METRICS].to_numpy()).all():
    raise ValueError("Fold metrics must be present and finite.")
if (data.groupby("region").size() < 2).any():
    raise ValueError("At least two valid folds are required for sample SD.")


In [ ]:
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6.5,
    "axes.labelsize": 7,
    "axes.linewidth": 0.6,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "svg.fonttype": "none",
    "savefig.facecolor": "white",
})
fig, axes = plt.subplots(
    3, 5, figsize=(WIDTH_MM / 25.4, HEIGHT_MM / 25.4), sharey=True,
)
fig.subplots_adjust(
    left=LEFT_MM / WIDTH_MM, right=1 - RIGHT_MM / WIDTH_MM,
    bottom=BOTTOM_MM / HEIGHT_MM, top=1 - TOP_MM / HEIGHT_MM,
    wspace=COLUMN_GAP_MM / PANEL_MM, hspace=ROW_GAP_MM / PANEL_MM,
)
x = np.arange(3)
point_count = 0

for panel, (ax, (region_id, name)) in enumerate(zip(axes.flat, REGIONS.items())):
    folds = data.loc[data["region"] == region_id]
    n_folds = folds["fold"].nunique()
    means = folds[METRICS].mean().to_numpy()
    stds = folds[METRICS].std(ddof=1).to_numpy()

    ax.set_box_aspect(1)
    ax.set_axisbelow(True)
    ax.grid(axis="y", color="#E7E7E7", linewidth=0.4)
    ax.bar(
        x, means, width=0.62,
        color=[to_rgba(color, 0.72) for color in COLORS],
        edgecolor=COLORS, linewidth=0.5, zorder=2,
    )
    ax.errorbar(
        x, means, yerr=stds, fmt="none", ecolor="#333333",
        elinewidth=0.65, capsize=1.7, capthick=0.65, zorder=3,
    )

    # Fixed horizontal offsets keep every fold visible and reproducible.
    # Fold order is preserved across all three metrics within each region.
    offsets = np.linspace(-0.20, 0.20, n_folds)
    for metric_index, metric in enumerate(METRICS):
        values = folds[metric].to_numpy()
        ax.scatter(
            metric_index + offsets, values,
            s=6.5, facecolors="none", edgecolors="#252525",
            linewidths=0.45, zorder=4,
        )
        point_count += len(values)

    ax.text(
        0.965, 0.955, f"{name} (n={n_folds})", transform=ax.transAxes,
        ha="right", va="top", fontsize=7, fontweight="bold",
    )
    ax.set_xticks(x, LABELS)
    ax.set_xlim(-0.55, 2.55)
    # The small lower margin preserves near-zero dots (e.g. EAF kappa).
    ax.set_ylim(-0.035, 1.025)
    ticks = np.linspace(0, 1, 6)
    ax.set_yticks(ticks, [f"{value:.1f}" for value in ticks])
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines["left"].set_bounds(0, 1)
    ax.spines["bottom"].set_position(("data", 0))
    ax.tick_params(axis="x", length=0, pad=4)
    ax.tick_params(axis="y", length=2, width=0.6, pad=2)
    if panel % 5 == 0:
        ax.set_ylabel("Metric value", labelpad=4)

FIGURES.mkdir(parents=True, exist_ok=True)
for extension in ('jpg', 'png', 'svg'):
    output_path = FIGURES / f'Supplementary-Figure-12.{extension}'
    # Do not crop with bbox_inches='tight': the canvas must remain 183 mm wide.
    fig.savefig(output_path, dpi=600)
    print(f'Saved {output_path}')

print(f'Canvas: {WIDTH_MM} × {HEIGHT_MM:.1f} mm; 15 panels, 45 bars, {point_count} fold scores.')
plt.show()
